This notebook (should be working) attempts to test the pipeline by just memorizing a single graffiti stroke.

The model generates the whole trajectory in 1-go and should output the same trajectory as the training trajectory.

`noisy traj * 0 -> MLP -> denoised traj -> noise prediction`

Because it can output the correct trajectory every time and the network isn't conditioned on anything, this isn't actually using diffusion, which is what notebook gerry06 is meant to rectify/test.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#@markdown ### **Imports**
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import math
import torch
import torch.nn as nn
import collections
import zarr
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv

In [ ]:
#@markdown ### **Dataset Demo**

# use cached dataset
dataset_path = "data/gml_000000.zarr"
# dataset_path = "data/gml_003000.zarr"

# parameters
pred_horizon = 132
obs_horizon = 131
action_horizon = 1
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)

print(np.argwhere(dataset.indices[:, 1] - dataset.indices[:, 0] == 132))
print(dataset.indices[256])
print(dataset.indices.shape)
# dataset.indices = dataset.indices[[256]]
dataset.indices = dataset.indices[dataset.indices[:, 0] == 301]
dataset.indices = dataset.indices[[-1,] * 50]
print(dataset.indices.shape)

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
print(len(list(iter(dataloader))))
batch = next(iter(dataloader))
print("batch['obs'].shape:", batch['obs'].shape)
print("batch['action'].shape", batch['action'].shape)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(batch['obs'][0, :, 0], batch['obs'][0, :, 1], 'ko-')
plt.plot(batch['action'][0, :, 0], batch['action'][0, :, 1], 'r.-')
# plt.plot(dataset.normalized_train_data['obs'][:, 0], dataset.normalized_train_data['obs'][:, 1], '.-')
# fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
# for i in range(3):
#     axes[i].plot(dataset[i + 13]['obs'][:, 0], dataset[i + 13]['obs'][:, 1], '.-')

In [ ]:
# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
#@markdown ### **Network Demo**

# observation and action dimensions corrsponding to
# the output of PushTEnv
obs_dim = 2
action_dim = 2

# create network object
noise_pred_net = MemorizationModel(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    noise_scheduler=noise_scheduler,
)

# example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, obs_horizon, obs_dim))
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# the noise prediction network
# takes noisy action, diffusion iteration and observation as input
# predicts the noise added to action
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# illustration of removing noise
# the actual noise removal is performed by NoiseScheduler
# and is dependent on the diffusion noise schedule
denoised_action = noised_action - noise

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

In [ ]:
#@markdown ### **Training**

num_epochs = 500

# Exponential Moving Average
# accelerates training and improves stability
# holds a copy of the model weights
ema = EMAModel(
    parameters=noise_pred_net.parameters(),
    model=noise_pred_net,
    power=0.75)

# Standard ADAM optimizer
# Note that EMA parametesr are not optimized
optimizer = torch.optim.AdamW(
    params=noise_pred_net.parameters(),
    lr=1e-4, weight_decay=1e-6)

# Cosine LR schedule with linear warmup
lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

with tqdm(range(num_epochs), desc='Epoch') as tglobal:
    # epoch loop
    for epoch_idx in tglobal:
        epoch_loss = list()
        # batch loop
        # with tqdm(dataloader, desc='Batch', leave=False) as tepoch:
        tepoch = dataloader
        if True:
            for nbatch in tepoch:
                # data normalized in dataset
                # device transfer
                nobs = nbatch['obs'].to(device)
                naction = nbatch['action'].to(device)
                B = nobs.shape[0]

                # observation as FiLM conditioning
                # (B, obs_horizon, obs_dim)
                obs_cond = nobs[:,:obs_horizon,:]
                # (B, obs_horizon * obs_dim)
                obs_cond = obs_cond.flatten(start_dim=1)

                # sample noise to add to actions
                noise = torch.randn(naction.shape, device=device)

                # sample a diffusion iteration for each data point
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (B,), device=device
                ).long()

                # add noise to the clean images according to the noise magnitude at each diffusion iteration
                # (this is the forward diffusion process)
                noisy_actions = noise_scheduler.add_noise(
                    naction, noise, timesteps)

                # predict the noise residual
                noise_pred = noise_pred_net(
                    noisy_actions, timesteps, global_cond=obs_cond)

                # L2 loss
                loss = nn.functional.mse_loss(noise_pred, noise)

                # optimize
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                # step lr scheduler every batch
                # this is different from standard pytorch behavior
                lr_scheduler.step()

                # update Exponential Moving Average of the model weights
                ema.step(noise_pred_net)

                # logging
                loss_cpu = loss.item()
                epoch_loss.append(loss_cpu)
                # tepoch.set_postfix(loss=loss_cpu)
        tglobal.set_postfix(loss=np.mean(epoch_loss))

# Weights of the EMA model
# is used for inference
# ema_noise_pred_net = ema.averaged_model

In [ ]:
# Visualize the noise prediction process by visualizing the noisy actions and the denoised actions
# (for the last iteration of training)

with torch.no_grad():
    unnoised_actions = compute_orig(noise_scheduler, timesteps, noisy_actions, noise_pred)
a = np.zeros((10, *noisy_actions.shape[1:]))
b = np.zeros((10, *noisy_actions.shape[1:]))
for i in range(10):
    ind = (timesteps - num_diffusion_iters * i / 9).abs().argmin().item()
    print(timesteps[ind].item(), end=', ')
    a[i] = noisy_actions[ind].cpu().numpy()
    b[i] = unnoised_actions[ind].cpu().numpy()
print()
fig, axes = plt.subplots(2, 5, figsize=(20, 5))
for i, ax in enumerate(axes.flatten()):
    ax.plot(a[9-i, :, 0], a[9-i, :, 1], 'r.-')
    ax.plot(b[9-i, :, 0], b[9-i, :, 1], 'b.-')

In [ ]:
ema_noise_pred_net = MemorizationModel(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
    noise_scheduler=noise_scheduler,
)
ema_noise_pred_net.to(device)
ema.copy_to(ema_noise_pred_net.parameters())

In [ ]:
#@markdown ### **Inference**

# limit enviornment interaction to 200 steps before termination
max_steps = 100
env = PaintingEnv(resolution=(512, 512))
# use a seed >200 to avoid initial states seen in the training dataset
# env.seed(100000)

# get first observation
obs = dataset[-1]['obs'][0]

# infer action
with torch.no_grad():
    B = 1
    # normalize observation
    nobs = dataset.normalize_obs(obs)
    # device transfer
    nobs = torch.from_numpy(nobs).to(device, dtype=torch.float32)
    nobs = nobs[None, :, None].expand(B, 2, obs_horizon)

    # reshape observation to (B,obs_horizon*obs_dim)
    obs_cond = nobs.unsqueeze(0).flatten(start_dim=1)

    # initialize action from Guassian noise
    noisy_action = torch.randn(
        (B, pred_horizon, action_dim), device=device)
    naction = noisy_action

    # init scheduler
    noise_scheduler.set_timesteps(num_diffusion_iters)

    tmp = []
    for k in noise_scheduler.timesteps:
        # predict noise
        noise_pred = ema_noise_pred_net(
            sample=naction,
            timestep=k,
            global_cond=obs_cond
        )

        # inverse diffusion step (remove noise)
        naction = noise_scheduler.step(
            model_output=noise_pred,
            timestep=k,
            sample=naction
        ).prev_sample

        tmp.append(naction.detach().to('cpu').numpy())
    # raise Exception

# unnormalize action
naction = naction.detach().to('cpu').numpy()
# (B, pred_horizon, action_dim)
naction = naction[0]
action_pred = dataset.unnormalize_action(naction)

In [ ]:
tmp2 = np.array(tmp)[:, 0, :, :]
tmp2.shape

In [ ]:
plt.plot(dataset[-1]['action'][:, 0], dataset[-1]['action'][:, 1], 'o-')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 5), sharex=True, sharey=True)
axes = axes.flatten()
for i in range(10):
    axes[i].plot(tmp2[i*11, :, 0], tmp2[i*11, :, 1], '.-')